# RescueVision - E9 FloodNet SegFormer B2

In [ ]:
import subprocess, sys, os, json, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = Path("/tmp/RescueVision")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/AliOmarAbdelhady/RescueVision.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))

def mark(stage, ok=True, extra=""):
    """Progress ledger in /kaggle/working: survives kills where the log does not."""
    rec = {"t": time.strftime("%H:%M:%S"), "stage": stage, "ok": bool(ok), "extra": str(extra)[:1500]}
    with open(WORK / "_progress.txt", "a") as f:
        f.write(json.dumps(rec) + "\n")
    (WORK / "_status.json").write_text(json.dumps(rec, indent=1))
    print("[mark]", rec, flush=True)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
mark("setup:start")
# pin albumentations: newer versions (preinstalled on the GPU image) have a
# stricter shape check that rejects our verified-consistent (HxWx3, HxW) pairs;
# --upgrade forces the downgrade past pip's "already satisfied"; all transforms
# ALSO use is_check_shapes=False as a version-proof second layer
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "albumentations==2.0.8"], check=False)
import albumentations as _A
mark("setup:albumentations", extra=_A.__version__)
print("albumentations", _A.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch"], check=False)
import segmentation_models_pytorch as smp
print("smp", smp.__version__)
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_workers=0: DataLoader workers died on the Kaggle GPU image; pin_memory
# off too (useless with workers=0, and a hard-kill vector on some drivers)
NW = 0
PIN = False
print("torch", torch.__version__, "| device:", DEVICE)
print(subprocess.run(["df", "-h", "/tmp", "/kaggle/working"], capture_output=True, text=True).stdout)
mark("setup:done", extra=str(DEVICE))

# per-epoch heartbeat: every trainer that logs an epoch also lands in the marks,
# so a silent kill still tells us how far training got and with what metrics
import rescuevision.training.logging_utils as _lu
_orig_log = _lu.CSVLogger.log
def _marked_log(self, metrics, epoch):
    try:
        mark(f"epoch{epoch}", extra=str({k: round(float(v), 4) for k, v in metrics.items()
                                         if isinstance(v, (int, float))})[:400])
    except Exception:
        pass
    return _orig_log(self, metrics, epoch)
_lu.CSVLogger.log = _marked_log
print("epoch heartbeat patch applied")

In [ ]:
import numpy as np, pandas as pd, random
from PIL import Image
from pathlib import Path

ROOT = None
for _c in Path("/kaggle/input").rglob("class_mapping.csv"):
    ROOT = _c.parent
    break
if ROOT is None and Path("/tmp/floodnet_src").exists():
    ROOT = Path("/tmp/floodnet_src")          # synthetic smoke path
assert ROOT is not None, "FloodNet dataset not found under /kaggle/input"
print("FloodNet root:", ROOT)
mark("floodnet:root", extra=str(ROOT))
TILES = Path("/tmp/floodnet_tiles")
(TILES / "train").mkdir(parents=True, exist_ok=True)
(TILES / "val").mkdir(parents=True, exist_ok=True)

from rescuevision.constants import FLOODNET_CLASSES, NUM_FLOODNET_CLASSES

import re
def norm_name(s):
    return re.sub(r"[^a-z]", "", str(s).lower())

NAME_TO_ID = {norm_name(v): k for k, v in FLOODNET_CLASSES.items()}

OFFICIAL_PALETTE = {  # official FloodNet colors, fallback if the CSV can't be parsed
    (0, 0, 0): 0, (72, 0, 255): 1, (255, 0, 0): 2, (0, 255, 255): 3, (255, 255, 0): 4,
    (0, 255, 0): 5, (0, 0, 255): 6, (255, 0, 255): 7, (0, 128, 128): 8, (128, 128, 128): 9,
}

color_to_id = {}
try:
    cm = pd.read_csv(ROOT / "class_mapping.csv")
    print("class_mapping.csv:\n", cm.to_string())
    cols = {c.lower(): c for c in cm.columns}
    name_col = next((cols[c] for c in cols if "name" in c or "class" in c), None)
    has_rgb = all(any(k in cols for k in (key,)) for key in ("r", "g", "b"))
    hex_col = next((cols[c] for c in cols if "hex" in c or "color" in c), None)
    for _, row in cm.iterrows():
        cid = NAME_TO_ID.get(norm_name(row[name_col])) if name_col else None
        if cid is None:
            continue
        if has_rgb:
            color_to_id[(int(row[cols["r"]]), int(row[cols["g"]]), int(row[cols["b"]]))] = cid
        elif hex_col and str(row[hex_col]).startswith("#"):
            h = str(row[hex_col]).lstrip("#")
            color_to_id[tuple(int(h[i:i+2], 16) for i in (0, 2, 4))] = cid
except Exception as e:
    print("could not parse class_mapping.csv:", e)
if len(color_to_id) < NUM_FLOODNET_CLASSES and (ROOT / "class_mapping.csv").exists() is False:
    print("no class_mapping.csv - masks assumed grayscale ids")
elif len(color_to_id) < NUM_FLOODNET_CLASSES:
    print("falling back to official FloodNet palette")
    color_to_id = OFFICIAL_PALETTE

def mask_to_ids(mask_rgb):
    # (H, W, 3) uint8 -> (H, W) class ids, nearest palette color
    m = mask_rgb.astype(np.int16)
    ids = np.zeros(m.shape[:2], dtype=np.uint8)
    best = np.full(m.shape[:2], 3 * 255, dtype=np.int16)
    for color, cid in color_to_id.items():
        d = (np.abs(m[:, :, 0] - color[0]) + np.abs(m[:, :, 1] - color[1])
             + np.abs(m[:, :, 2] - color[2])).astype(np.int16)
        better = d < best
        ids[better] = cid
        np.minimum(best, d, out=best)
    return ids

sample_mask = next((ROOT / "Train/Labeled/Flooded/mask").glob("*.png"))
sm = Image.open(sample_mask)
print("sample mask mode:", sm.mode, sm.size)
grayscale_mode = sm.mode in ("L", "P")
print("grayscale_mode:", grayscale_mode)
mark("floodnet:format", extra=f"grayscale={grayscale_mode}")

In [ ]:
import random, os
from tqdm.auto import tqdm

TILE = 512
SEED = 42
VAL_FRAC = 0.15
MAX_TRAIN_TILES = 16000    # quota-friendly caps (flood tiles kept with priority)
MAX_VAL_TILES = 2400
MAX_TMP_GB = 40       # /tmp safety guard

img_dirs = [ROOT / "Train/Labeled/Flooded", ROOT / "Train/Labeled/Non-Flooded"]
samples = []
for d in img_dirs:
    for img_path in sorted((d / "image").glob("*.jpg")):
        mask_path = d / "mask" / (img_path.stem + "_lab.png")   # official naming: {id}_lab.png
        if not mask_path.exists():
            mask_path = d / "mask" / (img_path.stem + ".png")
        if mask_path.exists():
            samples.append((img_path, mask_path))
print(f"{len(samples)} labeled images")
mark("floodnet:samples", extra=str(len(samples)))

rng = random.Random(SEED)
idx = rng.sample(range(len(samples)), max(1, int(len(samples) * VAL_FRAC)))
val_set = set(idx)

def tree_bytes(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())

records = []   # (img_npy, ids_npy, split, is_flood, bincount_row)
n_train = n_val = 0
for i, (img_path, mask_path) in enumerate(tqdm(samples, desc="tiling")):
    split = "val" if i in val_set else "train"
    img = np.array(Image.open(img_path).convert("RGB"))
    msk_img = Image.open(mask_path)
    if grayscale_mode:
        ids = np.array(msk_img.convert("L"), dtype=np.uint8)
    else:
        ids = mask_to_ids(np.array(msk_img.convert("RGB")))
    ids = np.clip(ids, 0, NUM_FLOODNET_CLASSES - 1)  # guard stray values
    H, W = img.shape[:2]
    for y in range(0, H - TILE + 1, TILE):
        for x in range(0, W - TILE + 1, TILE):
            tile_ids = ids[y:y+TILE, x:x+TILE]
            if len(np.unique(tile_ids)) < 2:
                continue  # skip fully-uniform tiles (usually all background)
            stem = f"{img_path.stem}_{y//TILE}_{x//TILE}"
            img_npy = TILES / split / f"{stem}_img.npy"
            ids_npy = TILES / split / f"{stem}_ids.npy"
            np.save(img_npy, img[y:y+TILE, x:x+TILE])
            np.save(ids_npy, tile_ids)
            cnt = np.bincount(tile_ids.flatten(), minlength=NUM_FLOODNET_CLASSES)
            records.append((img_npy, ids_npy, split, bool(cnt[1] + cnt[3] + cnt[5] > 0), cnt))
            if split == "train": n_train += 1
            else: n_val += 1
    if (i + 1) % 200 == 0 and tree_bytes(TILES) > MAX_TMP_GB * 1024**3:
        print(f"/tmp size guard hit at image {i+1}, stopping tiling early")
        mark("floodnet:tiling-guard", extra=str(i))
        break

print(f"tiles before cap: train={n_train} val={n_val}")

# cap train tiles: keep ALL flood tiles, subsample the rest deterministically
rng2 = np.random.default_rng(SEED)
train_recs = [r for r in records if r[2] == "train"]
if len(train_recs) > MAX_TRAIN_TILES:
    flood = [r for r in train_recs if r[3]]
    rest = [r for r in train_recs if not r[3]]
    budget = max(0, MAX_TRAIN_TILES - len(flood))
    keep = set(map(id, flood)) | set(map(id, rng2.choice(rest, size=min(budget, len(rest)), replace=False) if len(rest) else []))
    for r in train_recs:
        if id(r) not in keep:
            r[0].unlink(missing_ok=True); r[1].unlink(missing_ok=True)
    train_recs = [r for r in train_recs if id(r) in keep]
val_recs = [r for r in records if r[2] == "val"]
if len(val_recs) > MAX_VAL_TILES:
    keep = set(map(id, rng2.choice(val_recs, size=MAX_VAL_TILES, replace=False)))
    for r in val_recs:
        if id(r) not in keep:
            r[0].unlink(missing_ok=True); r[1].unlink(missing_ok=True)
    val_recs = [r for r in val_recs if id(r) in keep]

counts = np.sum([r[4] for r in train_recs], axis=0).astype(np.int64) if train_recs else np.zeros(NUM_FLOODNET_CLASSES, dtype=np.int64)
print(f"tiles after cap: train={len(train_recs)} val={len(val_recs)} | tree {tree_bytes(TILES)/1e9:.1f} GB")
print("class pixel counts:", counts.tolist())
assert counts.sum() > 0
mark("floodnet:tiles", extra=f"train={len(train_recs)} val={len(val_recs)}")

In [ ]:
import logging
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from rescuevision.models.losses import WeightedCEDiceLoss
from rescuevision.training.train_segmentation import train_segmentation, evaluate_segmentation
from rescuevision.utils.seed import set_seed
mark("floodnet:imports")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
OUTPUT = WORK
name = "floodnet_segformer_b2"
METRIC = "miou"

SEED = 42
AUG = {"horizontal_flip": 0.5, "vertical_flip": 0.5, "rotate90": 0.5, "brightness_contrast": 0.3, "normalize": True}
BS = 4
EPOCHS = 12
PATIENCE = 4

class FloodNetTileDataset(Dataset):
    def __init__(self, split_dir, transform=None):
        self.files = sorted(split_dir.glob("*_img.npy"))
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        base = str(self.files[i])[: -len("_img.npy")]   # strip the _img.npy suffix
        image = np.load(base + "_img.npy")
        mask = np.load(base + "_ids.npy")
        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"]
        mask = mask.long() if torch.is_tensor(mask) else torch.from_numpy(mask).long()
        return {"image": image, "mask": mask, "tile_id": Path(base).name}

from rescuevision.data.transforms import get_segmentation_transforms

set_seed(SEED)
train_ds = FloodNetTileDataset(TILES / "train", _nocheck(get_segmentation_transforms(AUG, "train", 512)))
val_ds = FloodNetTileDataset(TILES / "val", _nocheck(get_segmentation_transforms(AUG, "val", 512)))
print(f"train={len(train_ds)} val={len(val_ds)}")
mark("floodnet:data", extra=f"train={len(train_ds)}")
assert len(train_ds) > 0 and len(val_ds) > 0

train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=False, persistent_workers=bool(NW))
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW, pin_memory=False)

# recompute class counts from the extracted tiles (self-contained cell)
counts = np.zeros(NUM_FLOODNET_CLASSES, dtype=np.int64)
for _f in (TILES / "train").glob("*_ids.npy"):
    counts += np.bincount(np.load(_f).flatten(), minlength=NUM_FLOODNET_CLASSES)
print("class pixel counts:", counts.tolist())
assert counts.sum() > 0
freq = counts / counts.sum()
weights = 1.0 / np.sqrt(freq)
weights = weights / weights.sum() * NUM_FLOODNET_CLASSES
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
print("class weights:", [round(w, 3) for w in class_weights.tolist()])

# SMP SegFormer (mit_b2): the HF-transformers SegFormer hard-crashed CUDA on the
# Kaggle T4 image 4/4 times while SMP trained fine (building seg canary)
model = smp.Segformer(encoder_name="mit_b2", encoder_weights="imagenet",
                      in_channels=3, classes=NUM_FLOODNET_CLASSES).to(DEVICE)

loss_fn = WeightedCEDiceLoss(class_weights=class_weights, ce_weight=0.5, dice_weight=0.5, ignore_index=255)

config = {"seed": SEED, "device": DEVICE.type,
          "training": {"epochs": EPOCHS, "lr": 6e-5, "weight_decay": 1e-4, "scheduler": "cosine",
                       "mixed_precision": True, "early_stopping_patience": PATIENCE,
                       "save_best_metric": METRIC, "gradient_accumulation_steps": 2}}


# first-batch GPU probe: an OOM or crash here lands in the marks WITH a traceback
try:
    _it = iter(train_loader)
    for _ in range(2):
        _b = next(_it)
        _img = _b["image"].to(DEVICE)
        _m = _b["mask"].to(DEVICE)
        with torch.amp.autocast(DEVICE.type, enabled=True):
            _lg = model(_img)
            _loss = loss_fn(_lg.squeeze(1) if _lg.shape[1] == 1 else _lg,
                            _m.float() if _lg.shape[1] == 1 else _m)
        _loss.backward()
        model.zero_grad(set_to_none=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        mark("probe:ok", extra=f"peak={torch.cuda.max_memory_allocated()/1e9:.1f}GB")
    else:
        mark("probe:ok", extra="cpu")
except Exception:
    mark("probe:FAIL", extra=traceback.format_exc()[-1200:])
    raise
mark("floodnet:model-ready")
mark("floodnet:train-start", extra=f"train={len(train_ds)}")
best = train_segmentation(model, train_loader, val_loader, loss_fn, config,
                          output_dir=str(Path("/tmp/run") / name), is_binary=False,
                          num_classes=NUM_FLOODNET_CLASSES, ignore_index=255)
print("best:", best)
mark("floodnet:train-done", extra=str(best))

In [ ]:
import shutil as _shutil
_shutil.copytree(f"/tmp/run/floodnet_segformer_b2", WORK / "floodnet_segformer_b2", dirs_exist_ok=True)
print("results copied to", WORK / "floodnet_segformer_b2")
mark("copyout:floodnet_segformer_b2")
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

log_df = pd.read_csv(next((WORK / "floodnet_segformer_b2" / "logs").glob("*.csv")))
metric = "miou"
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log_df["epoch"], log_df["train_loss"], label="train")
if "val_loss" in log_df:
    axes[0].plot(log_df["epoch"], log_df["val_loss"], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(log_df["epoch"], log_df[metric], color="tab:green")
axes[1].set_title(f"val {metric} (best={log_df[metric].max():.4f} @ epoch {int(log_df.loc[log_df[metric].idxmax(), 'epoch'])})")
axes[1].set_xlabel("epoch"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(WORK / "floodnet_segformer_b2" / "training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import json

ckpt = torch.load(WORK / name / "checkpoints" / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(DEVICE).eval()
print(f"best checkpoint: epoch {ckpt['epoch']}, {METRIC}={ckpt.get(f'best_{METRIC}'):.4f}")

summary = {
    "run": name,
    "best_epoch": int(ckpt["epoch"]),
    f"best_val_{METRIC}": float(ckpt.get(f"best_{METRIC}")),
    "history_best": float(pd.read_csv(next((WORK / name / "logs").glob("*.csv")))[METRIC].max()),
}
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

import numpy as np
val_metrics = evaluate_segmentation(model, val_loader, DEVICE, is_binary=False,
                                    num_classes=NUM_FLOODNET_CLASSES, ignore_index=255)
print("VAL :", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in val_metrics.items()})
summary.update({"val": {k: round(v, 4) for k, v in val_metrics.items() if not isinstance(v, list)}})
with open(WORK / name / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

from rescuevision.constants import FLOODNET_CLASSES, FLOODNET_COLOR_MAP
import matplotlib.pyplot as plt
class_ious = val_metrics.get("class_ious")
if class_ious:
    for cid, iou in zip(range(NUM_FLOODNET_CLASSES), class_ious):
        print(f"  {FLOODNET_CLASSES[cid]:22s} IoU={iou:.4f}")

batch = next(iter(val_loader))
with torch.no_grad():
    pred = model(batch["image"].to(DEVICE)).argmax(1).cpu().numpy()
def colorize(m):
    rgb = np.zeros((*m.shape, 3), dtype=np.uint8)
    for c, col in FLOODNET_COLOR_MAP.items():
        rgb[m == c] = col
    return rgb
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i in range(min(4, batch["image"].shape[0])):
    img = batch["image"][i].numpy().transpose(1, 2, 0)
    img = (img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])).clip(0, 1)
    axes[0, i].imshow(img); axes[0, i].set_title(batch["tile_id"][i][:24])
    axes[1, i].imshow(colorize(batch["mask"][i].numpy())); axes[1, i].set_title("GT")
    axes[2, i].imshow(colorize(pred[i])); axes[2, i].set_title("pred")
    for ax in axes[:, i]: ax.axis("off")
plt.tight_layout(); plt.savefig(WORK / name / "val_predictions.png", dpi=120, bbox_inches="tight")
plt.show()
mark("floodnet:eval-done")